# Stage 2: Config-Driven & Logged Data Cleaning Pipeline
**AI/ML Capstone — Phase 1: Machine Learning Pipeline**  
**Dataset:** `data/raw/Combined_dataset.csv` → `data/processed/clean.parquet`  
**Objective:** Reproducible, config-driven data cleaning pipeline with comprehensive logging of every dropped or modified record.

---

### Cleaning Philosophy & Strict Safety Rules:
1. **Zero Raw Mutation**: The raw CSV remains untouched. All outputs are saved to `data/processed/` in Parquet format.
2. **Declarative & Config-Driven**: All thresholds, ranges, and exclusion lists reside in `src/cleaning_config.json`.
3. **No Silent Clipping**: Suspect measurements are not silently clamped; they are either converted to `NaN` (to be properly imputed inside training folds) or isolated into separate subsets.
4. **Extreme Outliers Preserved for Sensitivity Testing**: Values beyond domain norms (e.g. BOD > 100) are flagged via boolean columns rather than dropped immediately.
5. **Separation of Ambient vs Industrial/Waste Waters**: Effluent and Sewage are segregated to avoid mixing distinct environmental standards.

In [1]:
%load_ext autoreload
%autoreload 2

import os
import sys
import json
from pathlib import Path
from collections import OrderedDict
import numpy as np
import pandas as pd
from tabulate import tabulate

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", 35)
pd.set_option("display.width", 1000)

from src.cleaning import load_config, CleaningLog

config = load_config()
print("Cleaning configuration loaded successfully:")
print(json.dumps(config, indent=2))

Cleaning configuration loaded successfully:
{
  "seed": 42,
  "paths": {
    "raw_parquet": "data/processed/raw_cached.parquet",
    "clean_parquet": "data/processed/clean.parquet",
    "cleaning_log": "reports/02_cleaning_log.md",
    "cleaning_results": "reports/02_cleaning_results.json"
  },
  "duplicates": {
    "drop": true
  },
  "impossible_values": {
    "Ammonia (mg/l)": {
      "min": 0,
      "max": null,
      "action_below": "set_nan",
      "action_above": null,
      "reason_below": "Negative ammonia is physically impossible; likely below-detection-limit code or instrument artefact"
    },
    "Biochemical Oxygen Demand (mg/l)": {
      "min": 0,
      "max": null,
      "action_below": "set_nan",
      "action_above": null,
      "reason_below": "Negative BOD is physically impossible; likely instrument artefact"
    },
    "Dissolved Oxygen (mg/l)": {
      "min": 0.01,
      "max": 20,
      "action_below": "set_nan",
      "action_above": "set_nan",
      "reason_belo

## 1. Load Parquet Data & Initialize Cleaning Log
We load `data/processed/raw_cached.parquet` (2,827,977 rows) and initialize the `CleaningLog` tracker.

In [2]:
from src.stage1_analysis import load_data

df_raw = load_data()
log = CleaningLog(len(df_raw))
print(f"Initial raw rows: {log.initial_rows:,}")

Loading cached parquet: /Users/musthyalasadhvik/Desktop/projects/aml-capstone/AML_Capstone/data/processed/raw_cached.parquet
Initial raw rows: 2,827,977


## 2. Rule 1: Drop Exact Duplicates
Exact duplicate rows inflate sample size without providing new empirical information.

In [3]:
from src.cleaning import step1_drop_duplicates

df = step1_drop_duplicates(df_raw, log)

  [1_duplicates] Drop exact duplicate rows: 9,971 rows (0.353%) — dropped


## 3. Rule 2: Flag Impossible Values & Convert to NaN
Physical boundaries:
- pH outside $[0, 14]$
- Water temperature outside $[-2^\circ\text{C}, 40^\circ\text{C}]$
- Dissolved Oxygen $< 0.01\text{ mg/L}$
- Negative concentrations for Ammonia, BOD, Orthophosphate

Rather than dropping entire rows or silently clipping values, these measurements are flagged via boolean indicators (`flag_impossible_*`) and set to `np.nan`. Imputation is deferred to training pipelines.

In [4]:
from src.cleaning import step2_flag_impossible

df = step2_flag_impossible(df, config, log)

  [2_impossible_Ammonia (mg/l)_below] Ammonia (mg/l) < 0 → NaN: 2 rows (0.000%) — flagged + set to NaN
  [2_impossible_Biochemical Oxygen Demand (mg/l)_below] Biochemical Oxygen Demand (mg/l) < 0 → NaN: 2 rows (0.000%) — flagged + set to NaN
  [2_impossible_Dissolved Oxygen (mg/l)_below] Dissolved Oxygen (mg/l) < 0.01 → NaN: 204 rows (0.007%) — flagged + set to NaN
  [2_impossible_Orthophosphate (mg/l)_below] Orthophosphate (mg/l) < 0 → NaN: 10 rows (0.000%) — flagged + set to NaN
  [2_impossible_pH (ph units)_below] pH (ph units) < 0 → NaN: 1 rows (0.000%) — flagged + set to NaN
  [2_impossible_pH (ph units)_above] pH (ph units) > 14 → NaN: 504 rows (0.018%) — flagged + set to NaN
  [2_impossible_Temperature (cel)_below] Temperature (cel) < -2 → NaN: 1 rows (0.000%) — flagged + set to NaN
  [2_impossible_Temperature (cel)_above] Temperature (cel) > 40 → NaN: 7,284 rows (0.258%) — flagged + set to NaN


## 4. Rule 3: Detection-Limit Pile-Up Analysis
Sensor floors and detection limits create artificial spikes at values like $0.005$, $0.01$, or $0.02\text{ mg/L}$.  
We identify these clusters and verify they represent valid lower-bound readings that should be preserved.

In [5]:
import importlib
import src.cleaning
importlib.reload(src.cleaning)
from src.cleaning import step3_detection_limits

det_limits = step3_detection_limits(df, config, log)
print(tabulate(det_limits, headers="keys", tablefmt="github", showindex=False))

  [3_detection_limits] Detection-limit pile-ups identified in 4 columns: 359,419 rows (12.709%) — reported only (values kept)
| Column                |   Detection Limit Value |   Row Count |
|-----------------------|-------------------------|-------------|
| Ammonia (mg/l)        |                   0     |        1389 |
| Ammonia (mg/l)        |                   0.001 |        1362 |
| Ammonia (mg/l)        |                   0.002 |        2324 |
| Ammonia (mg/l)        |                   0.005 |        1837 |
| Ammonia (mg/l)        |                   0.01  |       10316 |
| Ammonia (mg/l)        |                   0.02  |       12821 |
| Ammonia (mg/l)        |                   0.05  |       14449 |
| Ammonia (mg/l)        |                   0.1   |        7675 |
| Orthophosphate (mg/l) |                   0     |         690 |
| Orthophosphate (mg/l) |                   0.001 |        6276 |
| Orthophosphate (mg/l) |                   0.002 |        2318 |
| Orthophosphate

## 5. Rule 4: Extreme Outlier Flagging
Rather than discarding severe outliers (e.g. Ammonia $>50\text{ mg/L}$, BOD $>100\text{ mg/L}$), we flag them via boolean columns (`outlier_*`). This preserves variance for subsequent sensitivity analysis in Stage 4.

In [6]:
from src.cleaning import step4_extreme_outliers

df, outlier_summary, iqr_report = step4_extreme_outliers(df, config, log)
print("### Outlier Summary:")
print(tabulate(outlier_summary, headers="keys", tablefmt="github", showindex=False))

  [4_outlier_Ammonia (mg/l)] Ammonia (mg/l) > 50 flagged as extreme: 6,658 rows (0.235%) — flag_only
  [4_outlier_Biochemical Oxygen Demand (mg/l)] Biochemical Oxygen Demand (mg/l) > 100 flagged as extreme: 21,242 rows (0.751%) — flag_only
  [4_outlier_Orthophosphate (mg/l)] Orthophosphate (mg/l) > 50 flagged as extreme: 302 rows (0.011%) — flag_only
  [4_outlier_Nitrate (mg/l)] Nitrate (mg/l) > 100 flagged as extreme: 1,009 rows (0.036%) — flag_only
### Outlier Summary:
| Ammonia (mg/l)   | Biochemical Oxygen Demand (mg/l)   | Orthophosphate (mg/l)   | Nitrate (mg/l)   |
|------------------|------------------------------------|-------------------------|------------------|
| threshold        | threshold                          | threshold               | threshold        |
| n_flagged        | n_flagged                          | n_flagged               | n_flagged        |
| pct              | pct                                | pct                     | pct              |


## 6. Rule 5: Target Designation
Verify that `CCME_Values` is designated as the primary continuous regression target. `CCME_WQI` is maintained as a categorical label for discrete evaluation.

In [7]:
from src.cleaning import step5_target_handling

df = step5_target_handling(df, config, log)

  [5_target] Target = CCME_Values (continuous). CCME_WQI is derived, not a target.: 0 rows (0.000%) — documented


## 7. Rule 6: Ambient Water Scope Filtering
Water samples from `Effluent` and `Sewage` sources are industrial / wastewater discharges rather than natural ambient waters.  
We segregate these 625,265 rows into a dedicated archive (`data/processed/excluded_non_ambient.parquet`) and proceed with ambient water bodies (Rivers, Lakes, Canals, Estuaries, etc.).

In [8]:
from src.cleaning import step6_waterbody_scope

df, df_excluded = step6_waterbody_scope(df, config, log)
print(f"Ambient water rows retained: {len(df):,}")
print(f"Excluded non-ambient rows segregated: {len(df_excluded):,}")

  [6_waterbody_scope] Exclude ['Effluent', 'Sewage'] — not ambient water: 625,265 rows (22.110%) — separated (excluded to separate file)
Ambient water rows retained: 2,192,741
Excluded non-ambient rows segregated: 625,265


## 8. Rule 7: Temporal Cutoff Filter
Observations prior to 1990 are sparse and reflect legacy analytical protocols. We filter records to $Year \ge 1990$.

In [9]:
from src.cleaning import step7_date_filter

df = step7_date_filter(df, config, log)

  [7_date_filter] Drop rows with year < 1990: 204,112 rows (7.218%) — dropped


## 9. Rule 8: Metadata Designation
Verify metadata columns (`Country`, `Area`, `Waterbody Type`, `Date`). These will be used for GroupKFold spatial/temporal validation splits rather than direct input features to prevent shortcuts.

In [10]:
from src.cleaning import step8_metadata_roles

df = step8_metadata_roles(df, config, log)

  [8_metadata_roles] Metadata columns: ['Country', 'Area', 'Waterbody Type', 'Date'] — NOT used as model features: 0 rows (0.000%) — documented


## 10. Pipeline Summary & Parquet Output Validation
Examine the complete cleaning log, final dataset shape, missingness statistics, and verify parity with `data/processed/clean.parquet`.

In [11]:
clean_log = log.to_dict()

print(f"Initial raw rows: {clean_log['initial_rows']:,}")
print(f"Final clean rows: {clean_log['final_rows']:,}")
print(f"Total dropped rows: {clean_log['total_dropped']:,} ({clean_log['pct_dropped']:.2f}%)")

# Detailed cleaning log table
log_df = pd.DataFrame(clean_log["steps"])[["rule", "description", "rows_affected", "pct_of_original", "action"]]
print(tabulate(log_df, headers="keys", tablefmt="github", showindex=False))

# Parquet file verification
clean_parquet_path = PROJECT_ROOT / "data" / "processed" / "clean.parquet"
if clean_parquet_path.exists():
    df_saved = pd.read_parquet(clean_parquet_path)
    print(f"Verified clean.parquet on disk: {df_saved.shape[0]:,} rows × {df_saved.shape[1]} columns")
    assert len(df_saved) == len(df), f"Row count mismatch! Pipeline: {len(df)}, File: {len(df_saved)}"
    print("Parquet integrity check: PASSED!")

Initial raw rows: 2,827,977
Final clean rows: 1,988,629
Total dropped rows: 839,348 (29.68%)
| rule                                                | description                                                                                  |   rows_affected |   pct_of_original | action                                |
|-----------------------------------------------------|----------------------------------------------------------------------------------------------|-----------------|-------------------|---------------------------------------|
| 1_duplicates                                        | Drop exact duplicate rows                                                                    |            9971 |            0.3526 | dropped                               |
| 2_impossible_Ammonia (mg/l)_below                   | Ammonia (mg/l) < 0 → NaN                                                                     |               2 |            0.0001 | flagged + set to NaN           

## 11. Conclusion & Stage 3 Readiness
The clean dataset `data/processed/clean.parquet` is now verified and ready for Stage 3 Exploratory Data Analysis (EDA):
- Total ambient, modern records: **1,988,629**
- Zero non-physical values
- Non-ambient discharges isolated to `data/processed/excluded_non_ambient.parquet`
- All decisions logged to `reports/02_cleaning_log.md` and configuration pinned in `src/cleaning_config.json`.